Notebook to generate synthetic goldens which will be (a) reviewed by a human and (b) used as a test set to evaluate the RAG chatbot.  

This notebook requires a Groq API key to generate the goldens.  

I am using a free model, so there are workarounds and quota-sensitive constraints, such as generating goldens from only a sample of the chunks using `generate_goldens_from_contexts()` rather than using `generate_goldens_from_docs()` and setting limits on the number of requests and time-delays.

In [0]:
%pip install \
    langchain==1.3.16 \
    langchain-chroma==1.1.0 \
    langchain-groq==1.1.3 \
    langchain-huggingface==1.2.2 \
    huggingface_hub==1.28.0 \
    sentence-transformers==6.0.0 \
    langchain_community \
    pypdf \
    torch==2.13.0 \
    torchvision==0.28.0 \
    torchaudio==2.11.0 \
    faiss-cpu==1.15.0 \
    deepeval \
    openai \
    dotenv
dbutils.library.restartPython()

In [0]:
import os
import time
from typing import List
from openai import OpenAI
from deepeval.models import DeepEvalBaseLLM
from dotenv import load_dotenv
from tests.custom_groq_model import CustomGroqModel


For generating goldens, I want to include chapter metadata on the chunks (so I hcan sample of to 5 chunks from each chapter, fewer if the chapter is short), and have longer chunks to pride a reasonable context. 

In [0]:
import re
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document
from collections import defaultdict

DATA_PATH = "fca_cobs_pdfs"

# load documents
loader = PyPDFDirectoryLoader(DATA_PATH)
docs = loader.load()
print(f"Loaded {len(docs)} pages from PDFs.")

# Cleaning function (this is improved over notebook 01, I need to refactor)
SKIP_EXACT = {'R', 'G', 'N', 'E', 'P', 'COBS', 'CHAPTER'}

def is_noise(line: str) -> bool:
    if not line or re.match(r'^[\.\s\-_]+$', line):
        return True
    if line.upper() in SKIP_EXACT:
        return True
    if 'www.handbook.fca.org.uk' in line:
        return True
    if re.search(r'(January|February|March|April|May|June|July|August|'
                 r'September|October|November|December)\s+\d{4}', line):
        return True
    if re.match(r'^(COBS|APP|SUP)\s+\d+[A-Z]?(\.\d+)*$', line, re.I):
        return True
    if len(line) < 60 and re.match(r'^(COBS|APP|SUP)\s+\d+[A-Z]?\s+\w', line, re.I):
        return True
    return False

def clean_fca_text(text: str) -> str:
    blocks = re.split(r'\n\s*\n', text)
    paras = []
    for block in blocks:
        lines = [ln.strip() for ln in block.split('\n')]
        kept = [ln for ln in lines if not is_noise(ln)]
        if kept:
            paras.append(' '.join(kept))
    return '\n\n'.join(paras)

# add chapter metadata and clean
docs_with_chapters = []
for doc in docs:
    source = doc.metadata.get("source", "")
    match = re.search(r'COBS\s+(\d+[A-Z]?)', source)
    
    if match:
        chapter_num = match.group(1)
        chapter_label = f"COBS {chapter_num}"
    else:
        chapter_label = "Unknown"
    
    cleaned_text = clean_fca_text(doc.page_content)
    
    docs_with_chapters.append(Document(
        page_content=cleaned_text,
        metadata={"source": source, "chapter": chapter_label}
    ))

# merge pages by chapter
chapter_texts = defaultdict(list)
for doc in docs_with_chapters:
    chapter_texts[doc.metadata["chapter"]].append(doc.page_content)

merged_docs = [
    Document(page_content="\n\n".join(texts), metadata={"chapter": chapter})
    for chapter, texts in chapter_texts.items()
]

# quick check
print("\nMerged Documents:")
for d in sorted(merged_docs, key=lambda x: len(x.page_content)):
    print(f"{len(d.page_content):6d} | {d.metadata['chapter']}")

Chunk the data, this time with 2000 characters.  The overlap is not really needed, but keeping it for now.

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from collections import Counter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", ". ", "\n", " ", ""], # prioritise sentence splits
)

# chunk merged documents
all_chunks = []
for doc in merged_docs:
    # split_documents returns a list of Document objects
    doc_chunks = text_splitter.split_documents([doc])
    all_chunks.extend(doc_chunks)

# filter out short chunks 
valid_chunks = [c for c in all_chunks if len(c.page_content) >= 800]

# quick check
print(f"Total chunks before filtering: {len(all_chunks)}")
print(f"Total valid chunks (>=800 chars): {len(valid_chunks)}")

# check distribution by chapter
chapter_counts = Counter(c.metadata["chapter"] for c in valid_chunks)
print("\nChunks per chapter:")
for chapter, count in sorted(chapter_counts.items()):
    print(f"{chapter:10s}: {count} chunks")

# check shortest valid chunk
if valid_chunks:
    sorted_by_len = sorted(valid_chunks, key=lambda x: len(x.page_content))
    shortest = sorted_by_len[0]
    print(f"\nShortest valid chunk: {len(shortest.page_content)} chars")
    print(f"Content snippet: {shortest.page_content[:100]}...")

Sample the chunks.  There are now fourteen chapters but a few only have 2 chunks.  I don't want to review too many goldens, so a maximum of 5 per chapter.

In [0]:
import random

random.seed(42)  

MAX_PER_CHAPTER = 5

by_chapter = defaultdict(list)
for chunk in valid_chunks:
    by_chapter[chunk.metadata["chapter"]].append(chunk)

sampled_chunks = []
for chapter, chunk_list in by_chapter.items():
    n = min(MAX_PER_CHAPTER, len(chunk_list))
    sampled_chunks.extend(random.sample(chunk_list, n))

print(f"Total sampled chunks: {len(sampled_chunks)}")
print(f"Chapters covered: {len(by_chapter)}")

# Check coverage
for chapter in sorted(by_chapter):
    count = min(MAX_PER_CHAPTER, len(by_chapter[chapter]))
    print(f"{chapter:10s}: {count} sampled of {len(by_chapter[chapter])}")

Add the Groq API key if necessary.

In [0]:
from getpass import getpass
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Initialise the DeepEval Synthesizer.  Note that it uses the CustomerGroqModel, since DeelEval's default is an OpenAI model.  I'm using the free model `groq/compund-mini` which in fact (as at 9 Sep 2026) integrates `llama-3.3-70b-versatile` and `openai/gpt-oss-120b`.

In [0]:
from deepeval.synthesizer import Synthesizer
from deepeval.synthesizer.config import ContextConstructionConfig, StylingConfig

synthesizer_llm = CustomGroqModel(
    model="groq/compound-mini", 
    seconds_delay=5, # slow the rate (free model)
    temperature=0.0,
    max_tokens=1000
)

styling_config = StylingConfig(
    input_format="Questions about the provided context",
    expected_output_format="Direct answer based on the context",
    task="Question Answering",
    scenario="Testing RAG accuracy"
)

synthesizer = Synthesizer(
    model=synthesizer_llm,
    styling_config=styling_config
)

The following loop has a lot of delays and re-tries as I was getting this working.  Right now I can manage to generate about 9-10 goldens before exhausting my daily quota with the free model.

In [0]:
import csv

def count_complete_rows(path):
    if not os.path.isfile(path):
        return 0
    with open(path, encoding="utf-8") as f:
        return sum(1 for row in csv.DictReader(f) if row.get("question"))


OUTPUT_CSV = "test_sets/synthetic_goldens.csv"
fieldnames = ["chunk_index", "chapter", "question", "answer", "evolutions", "synthetic_input_quality", "context_preview", "context"]
PAUSE = 75  # seconds between chunks


# build set of already-saved indices
saved_indices = set()
if os.path.isfile(OUTPUT_CSV):
    with open(OUTPUT_CSV, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row.get("question"):
                saved_indices.add(int(row["chunk_index"]))

print(f"{len(saved_indices)} valid goldens on disk: {sorted(saved_indices)}")

# header only if starting a new CSV
if not saved_indices:
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=fieldnames).writeheader()

with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    
    new_this_run = 0

    for i, chunk in enumerate(sampled_chunks, 1):
        if i in saved_indices:
            continue  # already done

        success = False
        for attempt in range(3):
            try:
                goldens = synthesizer.generate_goldens_from_contexts(
                    contexts=[[chunk.page_content]],
                    max_goldens_per_context=1
                )
                if not goldens:
                    print(f"ALERT {i}: no golden")
                    break

                g = goldens[0]
                ctx = " ".join(g.context) if isinstance(g.context, list) else str(g.context)

                writer.writerow({
                    "chunk_index": i,
                    "chapter": chunk.metadata.get("chapter", "Unknown"),
                    "question": g.input,
                    "evolutions": g.additional_metadata["evolutions"],
                    "synthetic_input_quality": g.additional_metadata["synthetic_input_quality"],
                    "answer": g.expected_output,
                    "context_preview": ctx[:150] + "...",
                    "context": ctx,
                })
                f.flush()
                os.fsync(f.fileno())
                print(f"{i}/{len(sampled_chunks)} {chunk.metadata.get('chapter')}")
                success = True
                new_this_run += 1
                break

            except Exception as e:
                wait = PAUSE * (attempt + 1)
                print(f"ERROR {i} attempt {attempt + 1}: {e} — waiting {wait}s")
                time.sleep(wait)

        if not success:
            print(f"SKIPPED {i}: skipped after 3 attempts")
            
        if new_this_run >= 10:
            print("Stopping after 10 — quota safety")
            break

        time.sleep(PAUSE)

The code cells below here include various quicks tests and variations I tried, leaving here for now.

In [0]:
# synthesizer_llm = ChatGroq(
#     model="groq/compound-mini", 
#     temperature=0.0,
#     max_tokens=1000
# )

In [0]:
print(sampled_chunks[0].page_content)

In [0]:
for i, chunk in enumerate(sampled_chunks[20:22], 1):
    chunk_text = chunk.page_content
    chapter = chunk.metadata.get("chapter", "Unknown")
    print(chunk_text)
    print(chapter)

In [0]:
# test a single chunk
chunk_text = sampled_chunks[0].page_content
contexts = [[chunk_text]]
goldens = synthesizer.generate_goldens_from_contexts(
    contexts=contexts,
    max_goldens_per_context=1
)

In [0]:
print(goldens)

In [0]:

for i, context in enumerate(sampled_chunks[0:2]):
    print(context)

In [0]:
from groq import Groq
client = Groq()
completion = client.chat.completions.create(
    # model="openai/gpt-oss-120b",
    model="llama-3.3-70b-versatile"
    messages=[
        {
            "role": "user",
            "content": "Explain why fast inference is critical for reasoning models"
        }
    ]
)
print(completion.choices[0].message.content)

In [0]:
from langchain_groq import ChatGroq
try:
    llm = ChatGroq(model_name="groq/compound-mini", temperature=0)
    response = llm.invoke("Hello")
    print("Groq connection successful!")
    print(response.content)
except Exception as e:
    print(f"Groq connection failed: {e}")

In [0]:
custom_groq_model = CustomGroqModel(seconds_delay=15)
print(custom_groq_model.generate("Write me a joke"))